# 04 - Export Tableau Public

Objectif : preparer le fichier final pour le tableau de bord Tableau Public.

In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

cleaned_path = project_root / 'data' / 'cleaned' / 'buildings_clean.csv'
dashboard_path = project_root / 'dashboard' / 'building_energy_performance_dashboard.csv'

## Creer le fichier dashboard

In [2]:
df = pd.read_csv(cleaned_path)

dashboard_columns = [
    'building_id', 'data_year', 'building_name', 'building_type', 'neighborhood',
    'year_built', 'number_of_floors', 'property_gfa_total', 'energy_star_score',
    'site_eui', 'site_energy_use', 'electricity_kwh', 'natural_gas_therms',
    'total_ghg_emissions', 'ghg_emissions_intensity', 'compliance_status',
    'energy_star_score_was_missing', 'natural_gas_was_missing'
]

dashboard_df = df[dashboard_columns].copy()

dashboard_df = dashboard_df.assign(
    building_age=dashboard_df['data_year'] - dashboard_df['year_built'],
    energy_use_per_floor=dashboard_df['site_energy_use'] / dashboard_df['number_of_floors'],
    energy_per_sqft=dashboard_df['site_energy_use'] / dashboard_df['property_gfa_total'],
    energy_star_category=pd.cut(
        dashboard_df['energy_star_score'],
        bins=[0, 50, 75, 100],
        labels=['A verifier', 'Moyen', 'Bon'],
        include_lowest=True
    )
)

dashboard_df = dashboard_df.assign(
    priority_score=(
        dashboard_df['site_energy_use'].rank(pct=True) * 0.4
        + dashboard_df['total_ghg_emissions'].rank(pct=True) * 0.4
        + (100 - dashboard_df['energy_star_score']).rank(pct=True) * 0.2
    ).round(4)
)

dashboard_df.head()

,building_id,data_year,building_name,building_type,neighborhood,year_built,number_of_floors,property_gfa_total,energy_star_score,site_eui,...,total_ghg_emissions,ghg_emissions_intensity,compliance_status,energy_star_score_was_missing,natural_gas_was_missing,building_age,energy_use_per_floor,energy_per_sqft,energy_star_category,priority_score
0,1,2024,Mayflower Park Hotel,Nonresidential,Downtown,1927,12,88434,59.0,61.700001,...,263.3,2.98,Not Compliant,0,0,97,5.928298e+05,80.443698,Moyen,0.8652
1,2,2024,Paramount Hotel,Nonresidential,Downtown,1996,11,103566,85.0,71.500000,...,208.6,2.36,Compliant,0,0,28,5.755149e+05,61.126856,Bon,0.7478
2,3,2024,Westin Hotel (Parent Building),Nonresidential,Downtown,1969,41,956110,71.0,81.699997,...,2418.2,3.18,Compliant,0,0,55,1.649104e+06,70.717035,Moyen,0.9390
3,5,2024,Hotel Max,Nonresidential,Downtown,1926,10,61320,50.0,86.000000,...,190.1,3.10,Compliant,0,0,98,5.273981e+05,86.007518,A verifier,0.8398
4,8,2024,Warwick Seattle Hotel,Nonresidential,Downtown,1980,18,175580,87.0,97.099998,...,417.9,3.68,Compliant,0,0,44,6.657755e+05,68.253554,Bon,0.8049


## Verifier et exporter

In [3]:
dashboard_path.parent.mkdir(parents=True, exist_ok=True)
dashboard_df.to_csv(dashboard_path, index=False)

print(f'Fichier exporte : {dashboard_path}')
print(f'Lignes : {len(dashboard_df):,}')
print(f'Colonnes : {dashboard_df.shape[1]}')
print(f'Valeurs manquantes : {dashboard_df.isna().sum().sum()}')

Fichier exporte : c:\Users\djami\Desktop\Projects\Energy-Performance-Dashboard\dashboard\building_energy_performance_dashboard.csv
Lignes : 33,268
Colonnes : 23
Valeurs manquantes : 0
